In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

test = pd.read_csv("data/idsc_test.csv")

In [2]:
test["data"] = pd.to_datetime(test["dt_dep"])

In [3]:
import requests

def download_images(df):
    unlock = False
    for _, row in df.iterrows():
        data = row['data']
        path = row['path']
        
        # Extrai o nome do arquivo da URL
        nome_arquivo = path.split('/')[-1]
        print(nome_arquivo)

        # Define o caminho onde a imagem será salva
        pasta_destino = f'data/img/testtesttest/'
        
        # Faz o download da imagem usando a biblioteca requests
        response = requests.get(path)
        
        # Verifica se a resposta foi bem sucedida (código 200)
        if response.status_code == 200:
            with open(f'{pasta_destino}{data}.jpg', 'wb') as file:
                file.write(response.content)
                print(f"Imagem {nome_arquivo} salva em {pasta_destino}")
        else:
            print(f"Erro ao baixar a imagem {nome_arquivo}")


In [4]:
#download_images(test)

In [25]:
import os
import subprocess
import pandas as pd
import json
# Cria o arquivo de log
log_file = open('log/log2.txt', 'w')

# Cria o dataset Pandas

data_list = []
#dataset = pd.DataFrame(columns=['data','X','Y','poly','vertice'])

# Percorre cada arquivo na pasta 'data/img/'
for file in os.listdir('data/img/test/'):
    # Obtém o caminho completo do arquivo
    file_path = os.path.join('data/img/test/', file)
    
    # Chama o script 'utils/decodeImage.py' passando o caminho da imagem como parâmetro
    try:
        output = subprocess.check_output(['python3', 'utils/decodeImage.py', file_path])
        output = json.loads(output)
        
        # Faz o processamento necessário no 'output' para extrair as informações desejadas
        # Supondo que as informações extr
        # Supondo que as informações extraídas estejam no formato mencionado no exemplo
        # Faz o parse do output e extrai as coordenadas e polígonos
        polyCount = 0
        for poly in output:
            verticeCount = 0
            for vertice in poly:
                x = vertice[0][0]
                y = vertice[0][1]
            
            # Adiciona os dados no dataset Pandas
                data_list.append({'data': file,
                                          'X': x, 
                                          'Y': y,
                                          'poly': polyCount,
                                          'vertice': verticeCount})
                verticeCount = verticeCount + 1
            polyCount = polyCount + 1
        # Registra no arquivo de log que não ocorreu erro na execução
        log_file.write(f'{file}: sem erros\n')
    except subprocess.CalledProcessError as err:
        # Registra no arquivo de log que ocorreu um erro na execução
        log_file.write(f'{file}: erro na execução - {err}\n')
    
dataset = pd.DataFrame(data_list)
# Fecha o arquivo de log
log_file.close()

# Exibe o dataset Pandas
print(dataset)
dataset.to_csv('data/testImageData.csv', index=False)


                           data     X     Y  poly  vertice
0       2023-05-25 01:58:06.jpg  1151  1303     0        0
1       2023-05-25 01:58:06.jpg  1149  1373     0        1
2       2023-05-25 01:58:06.jpg   867  1451     0        2
3       2023-05-25 01:58:06.jpg   531  1072     0        3
4       2023-05-25 01:58:06.jpg   528  1010     0        4
...                         ...   ...   ...   ...      ...
354372  2023-05-28 18:08:04.jpg   630     0   755        3
354373  2023-05-28 18:08:04.jpg  1104  1449   756        0
354374  2023-05-28 18:08:04.jpg  1104  1451   756        1
354375  2023-05-28 18:08:04.jpg  1108  1451   756        2
354376  2023-05-28 18:08:04.jpg  1108  1449   756        3

[354377 rows x 5 columns]


In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("local_spark") \
        .config('spark.jars.packages', 
        'org.datasyslab:geospark:1.3.1,org.datasyslab:geospark-sql_2.3:1.3.1'
    ).getOrCreate()


from geospark.register import GeoSparkRegistrator
GeoSparkRegistrator.registerAll(spark)



23/10/04 23:00:17 WARN Utils: Your hostname, bruno-Inspiron-7572 resolves to a loopback address: 127.0.1.1; using 192.168.1.12 instead (on interface wlp3s0)
23/10/04 23:00:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/bruno/.local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/bruno/.ivy2/cache
The jars for the packages stored in: /home/bruno/.ivy2/jars
org.datasyslab#geospark added as a dependency
org.datasyslab#geospark-sql_2.3 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ccea02a8-cd79-4370-b688-b9eeb44e011b;1.0
	confs: [default]
	found org.datasyslab#geospark;1.3.1 in central
	found org.datasyslab#geospark-sql_2.3;1.3.1 in central
:: resolution report :: resolve 388ms :: artifacts dl 12ms
	:: modules in use:
	org.datasyslab#geospark;1.3.1 from central in [default]
	org.datasyslab#geospark-sql_2.3;1.3.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	----

In [ ]:
cat62 = spark.read.csv('data/cat-62.csv', inferSchema=True, header=True)
